# Motorsport Strategy Lab — F1 / WEC / IMSA race-strategy engine

**No fabricated data.** Every number in this notebook is measured from real timing data (FastF1 for F1, a community-maintained multi-series DuckDB for WEC/IMSA, plus a public F1 per-lap history for cross-calendar breadth) — nothing is simulated-as-if-real or assumed.

Source: https://github.com/mohammedmedjadj/Motorsport-Strategy-Lab

This notebook clones the repo and runs against the **committed derived data already in git** — no external dataset attachment required for the core sections. One section optionally uses the public Kaggle F1 dataset if you attach it (instructions inline).

**What this project does, in one line per layer:**
- **Degradation** — how much a tyre slows per lap, fitted per race with driver/stint fixed effects.
- **Neutralisation calibration** — out-of-sample Safety Car / FCY probability, scored with proper scoring rules (Brier, log-loss) against a positive control.
- **Multi-stop simulator** — an exact dynamic program over the whole race (not just the next stop), Monte Carlo race-time distributions with common random numbers.
- **Adversarial rival** — the pit-stop decision as a two-player game (a rival that covers, not a fixed plan).
- **Reliability / attrition** — probability of reaching the classified finish, from 13 seasons of WEC results and 14 of F1.
- **Retrospective audit** — does real race-winning behaviour corroborate the model's own conclusions?

In [ ]:
!git clone --quiet https://github.com/mohammedmedjadj/Motorsport-Strategy-Lab.git
%cd Motorsport-Strategy-Lab

In [ ]:
# Minimal, exact dependencies for this notebook — no GPU, no PyMC: nothing in
# this codebase currently uses either, and none of these sections benefit from
# a GPU (the Monte Carlo code is plain vectorised NumPy on modest draw counts,
# not a GPU-bound workload).
!pip install --quiet pandas numpy scipy matplotlib duckdb

In [ ]:
import sys
sys.path.insert(0, ".")

import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 110

## 1. Reliability — who actually finishes

Finishing is itself a strategy variable in endurance racing. Rates below use a Jeffreys `Beta(0.5, 0.5)` smoother, so a thin group gets a wide interval instead of a false 0%/100%.

In [ ]:
wec_rel = pd.read_csv("data/derived/wec/reliability.csv")
f1_rel = pd.read_csv("data/derived/f1/reliability.csv")

wec_duration = wec_rel[wec_rel["dimension"] == "duration"].sort_values("group")
display(wec_duration)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
order = ["4h", "6h", "8h", "12h", "24h"]
d = wec_duration.set_index("group").reindex(order).dropna()
ax[0].bar(d.index, d["finish_rate"], color="#2563eb")
ax[0].errorbar(d.index, d["finish_rate"], yerr=[d["finish_rate"] - d["lo95"], d["hi95"] - d["finish_rate"]],
               fmt="none", ecolor="black", capsize=3)
ax[0].set_title("WEC finish rate vs race duration\n(13 seasons, 2011-2023)")
ax[0].set_ylabel("finish rate")

f1_era = f1_rel[f1_rel["dimension"] == "era"].sort_values("finish_rate")
ax[1].barh(f1_era["group"], f1_era["finish_rate"], color="#dc2626")
ax[1].set_title("F1 finish rate by regulation era\n(2011-2024, 5 980 entries)")
ax[1].set_xlabel("finish rate")
plt.tight_layout()
plt.show()

print(f"WEC: a 24h race finishes at {d.loc['24h','finish_rate']:.0%}, "
      f"a 6h race at {d.loc['6h','finish_rate']:.0%} — attrition rises with "
      f"duration, the model's own positive control.")

## 2. Multi-stop strategy across 21 real circuits

`src/simulator/multistop.py` plans the *whole* race with an exact dynamic program — not just the next stop — minimising green running time + tyre degradation + `n_stops × pit_loss` over every partition of the race into fuel-length stints. Run across every circuit the source verifiably carries (widened from 4 hand-picked circuits to all 21 eligible ones, `scripts/discover_endurance_events.py`).

In [ ]:
plans = pd.read_csv("data/derived/endurance/multistop_plans.csv")
display(plans[["series", "circuit", "year", "race_laps", "net_slope_s",
               "min_stops", "optimal_stops", "fuel_limited", "breakeven_slope_s"]])

n_fuel_limited = plans["fuel_limited"].sum()
print(f"\n{n_fuel_limited}/{len(plans)} circuits: the optimum never takes more "
      f"stops than the fuel minimum — no scoped race is tyre-limited on stop "
      f"count. Where a break-even slope exists, measured degradation would "
      f"need to be {plans['slope_headroom_x'].min():.1f}x to "
      f"{plans['slope_headroom_x'].max():.1f}x steeper before an extra stop paid off.")

## 3. Prove it live: refit one real race end to end

Not just reading a CSV — this cell reloads the committed 2024 WEC Bahrain laps, fits degradation, fits the neutralisation posterior, and solves the exact stop-count optimisation and Monte Carlo distribution from scratch, right here.

In [ ]:
from src.data.endurance_loader import EnduranceLoader
from src.degradation.endurance import build_endurance_frame, fit_endurance_degradation
from src.safety_car.endurance import (
    extract_events, fit_neutralisation_models, load_race_flags, race_timeline,
)
from src.simulator.endurance import build_race_model
from src.simulator.multistop import optimal_stop_plan, evaluate_plan, min_stops_plan

SERIES, YEAR, EVENT, CLASS = "wec", 2024, "Bahrain", "HYPERCAR"

laps = EnduranceLoader(SERIES).load_laps(YEAR, EVENT, CLASS)
fit = fit_endurance_degradation(build_endurance_frame(laps))

timeline = race_timeline(load_race_flags())
events = extract_events(timeline)
posteriors = {(m.series, m.kind): m for m in fit_neutralisation_models(timeline, events)}
fcy, sc = posteriors[(SERIES, "FCY")], posteriors[(SERIES, "SC")]
fcy_dur = tuple(e.duration_laps for e in events if e.series == SERIES and e.kind == "FCY")
sc_dur = tuple(e.duration_laps for e in events if e.series == SERIES and e.kind == "SC")

model = build_race_model(
    laps, fit.net_slope.value, fit.net_slope.se,
    fcy.n_events + 0.5, fcy.laps_exposure, fcy_dur, fit.rmse_s,
    sc_alpha=sc.n_events + 0.5, sc_exposure=sc.laps_exposure, sc_durations=sc_dur,
)
race_laps = int(laps.groupby("car")["lap"].max().median())

opt = optimal_stop_plan(race_laps, model.green_pace_s, model.net_slope_s,
                         model.pit_loss_s, model.fuel_range_laps)
naive = min_stops_plan(race_laps, model.fuel_range_laps)
dist = evaluate_plan(opt, race_laps, model, n_draws=4000)

print(f"{EVENT} {YEAR}: {race_laps} laps, green pace {model.green_pace_s:.1f}s, "
      f"net slope {model.net_slope_s:+.4f}s/lap, fuel range {model.fuel_range_laps} laps")
print(f"Fuel-minimum stops: {naive.n_stops}  |  DP optimum: {opt.n_stops} "
      f"(stint plan {opt.stint_lengths})")
print(f"Race-time distribution (Monte Carlo, 4 000 draws, common random "
      f"numbers): median {dist['median_s']:.0f}s, P10-P90 "
      f"[{dist['p10_s']:.0f}, {dist['p90_s']:.0f}]s")

## 4. Retrospective audit: did real winners run fuel-limited?

The model above concludes every scoped race is fuel-limited on stop count. Tested against **what winners actually did** — their real fuel-stint lengths, reconstructed from committed laps, compared to the measured fuel range.

In [ ]:
audit = pd.read_csv("data/derived/endurance/fuel_limited_audit.csv")
by_series = audit.groupby("series")["ran_fuel_limited"].agg(["sum", "count", "mean"])
display(by_series)

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(by_series.index, by_series["mean"], color=["#dc2626", "#2563eb"])
ax.set_ylim(0, 1)
ax.set_ylabel("share of winners running a full-fuel-range stint")
ax.set_title("Retrospective corroboration: real winners vs the model")
for i, (n, tot) in enumerate(zip(by_series["sum"], by_series["count"])):
    ax.text(i, by_series["mean"].iloc[i] + 0.02, f"{int(n)}/{int(tot)}", ha="center")
plt.show()

print(f"{int(audit['ran_fuel_limited'].sum())}/{len(audit)} audited winners "
      f"across every scoped race ran fuel-limited — real winning behaviour "
      f"corroborates the model, not just a self-consistent simulation.")

## 5. Optional: F1 full-calendar breadth (attach the public F1 dataset)

The high-fidelity FastF1 model covers 4 circuits deeply (tyre compound, per-lap Safety-Car flags). A breadth layer separately fits net degradation across **35 F1 circuits since 2011** from a public per-lap Kaggle export, and — because F1 has had no in-race refuelling since 2010 — **separates tyre wear from fuel burn**, something the endurance data structurally cannot do (every endurance stop refuels).

This section needs that dataset attached as a Kaggle input (`Add Data` → search for a Formula 1 per-lap results dataset carrying `lap_times.csv`, `pit_stops.csv`, `races.csv`, `circuits.csv`, `status.csv` — verify the exact slug on Kaggle, as it is not redistributed by this repo). It degrades gracefully if not attached.

In [ ]:
import glob

candidates = glob.glob("/kaggle/input/*/lap_times.csv")
if not candidates:
    print("No F1 per-lap dataset attached — skipping this section.\n"
          "To run it: Add Data -> attach a public Kaggle F1 results dataset "
          "carrying lap_times.csv/pit_stops.csv/races.csv/circuits.csv/status.csv, "
          "then set F1_EXTERNAL below to its /kaggle/input/... folder and re-run.")
else:
    import src.data.f1_history_loader as f1_hist
    f1_hist.F1_EXTERNAL = __import__("pathlib").Path(candidates[0]).parent
    from src.degradation.f1_history import fit_history_degradation

    df = f1_hist.load_f1_lap_history(era_start=2022)
    table = fit_history_degradation(df)
    ge = table.dropna(subset=["tyre_slope_s"])
    print(f"Fitted {ge['circuit'].nunique()} circuits, {len(ge)} race-seasons.")
    print(f"Isolated tyre wear positive in {(ge['tyre_slope_s'] > 0).mean():.0%} "
          f"of races (the net slope alone is often near-zero or negative — "
          f"fuel-burn masks it — this two-regressor fit recovers it).")
    display(ge.sort_values("tyre_slope_s", ascending=False).head(8)
            [["circuit", "year", "tyre_slope_s", "fuel_evo_slope_s", "net_slope_s"]])

## Methodology notes

- **No fabricated data anywhere.** Every gap in a source (missing weather, zero observed Safety Cars, no per-lap flags for a specific race) is measured and reported, never imputed.
- **No data leakage.** Neutralisation calibration is scored strictly out-of-sample (leave-one-race-out).
- **Uncertainty is first-class.** Every rate carries a credible interval; every simulated race a full distribution, not a point estimate.
- **Nothing is pooled across regulation eras** — F1's 2026 rules (new power units, active aero, lighter/narrower cars) are walled off from every historical fit.
- Full methodology, all reports, and every test: https://github.com/mohammedmedjadj/Motorsport-Strategy-Lab

Feedback welcome — open an issue on the repo or comment below.